In [ ]:
import sys
sys.path.append('../')

import importlib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage

from src.preprocessing import (
    load_cluster_data, clean_customer_data,
    engineer_customer_features, encode_categorical, scale_features
)
import src.utils as _utils
importlib.reload(_utils)
from src.utils import show_figure
import src.display as _display
importlib.reload(_display)
from src.display import (
    init_notebook_theme, show_hero, show_section, show_insight,
    show_info, show_warning, show_success, show_metrics_row,
    show_findings_list, show_badge, show_table_html, show_architecture_card,
)
init_notebook_theme()

DATA_PATH = '../data/raw/data_cluster.csv'
RANDOM_STATE = 42


In [ ]:
show_hero(
    title="Exercice 2 — Segmentation intelligente des clients",
    objective="Identifier des profils clients pour optimiser les campagnes marketing.",
    plan_items=[
        "EDA", "Nettoyage & features", "Prétraitement clustering",
        "Clustering multi-algos", "Évaluation", "Interprétation métier", "Recommandations",
    ],
)
show_section("1. Analyse exploratoire (EDA)")

In [ ]:
df = load_cluster_data(DATA_PATH)
show_metrics_row({"Lignes": f"{df.shape[0]:,}", "Colonnes": len(df.columns)})
print(df.columns.tolist())

In [ ]:
print("--- Informations ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n--- Statistiques ---")
df.describe()

In [ ]:
from src.charts.segmentation import build_distributions
from src.utils import show_figure

fig = build_distributions(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_distributions.png', width=1100, height=650)


In [ ]:
from src.charts.segmentation import build_categorical
from src.utils import show_figure

fig = build_categorical(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_categorical.png', width=1000, height=450)


In [ ]:
from src.charts.segmentation import build_spending_channels
from src.utils import show_figure

fig = build_spending_channels(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_spending_channels.png', width=1000, height=450)


In [ ]:
show_section("2. Nettoyage & Feature Engineering")

In [ ]:
df_clean = clean_customer_data(df)
df_clean = engineer_customer_features(df_clean)

print(f"Lignes avant nettoyage : {len(df)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"\nNouvelles features créées : Age, Children, TotalSpend, TotalPurchases")

In [ ]:
from src.charts.segmentation import build_correlation
from src.utils import show_figure

fig = build_correlation(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_correlation.png', width=1000, height=800)


In [ ]:
show_section("3. Prétraitement pour clustering")

In [ ]:
# Sélection des features pour le clustering
cat_cols = ['Education', 'Marital_Status']
drop_cols = ['ID', 'Dt_Customer', 'Year_Birth'] + cat_cols

df_encoded = encode_categorical(df_clean, cat_cols)
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

X = df_encoded[feature_cols].copy()
print(f"Features pour clustering ({len(feature_cols)}) :\n{feature_cols}")

In [ ]:
X_scaled, scaler = scale_features(X)

# PCA pour visualisation 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.1%}")

# PCA pour réduction dimensionnelle (95% de variance)
pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)
print(f"Composantes pour 95% variance : {X_pca_full.shape[1]}")

In [ ]:
from src.charts.segmentation import build_pca_scree
from src.utils import show_figure

fig = build_pca_scree(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_pca_scree.png', width=1000, height=450)


In [ ]:
show_section("4. Clustering")

In [ ]:
from src.charts.segmentation import build_kmeans_selection
from src.utils import show_figure

fig = build_kmeans_selection(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_kmeans_selection.png', width=1200, height=450)


In [ ]:
# K-Means final
kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca_full)

# Clustering hiérarchique
agglo = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
agglo_labels = agglo.fit_predict(X_pca_full)

# GMM
gmm = GaussianMixture(n_components=BEST_K, random_state=RANDOM_STATE, covariance_type='full')
gmm_labels = gmm.fit_predict(X_pca_full)

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_pca_full)
n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"DBSCAN : {n_dbscan_clusters} clusters, {(dbscan_labels == -1).sum()} outliers")

In [ ]:
from src.charts.segmentation import build_clustering_comparison
from src.utils import show_figure

fig = build_clustering_comparison(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_clustering_comparison.png', width=1000, height=800)


In [ ]:
from src.charts.segmentation import build_dendrogram
from src.utils import show_figure

fig = build_dendrogram(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_dendrogram.png', width=1200, height=500)


In [ ]:
show_section("5. Évaluation & comparaison")

In [ ]:
# Tableau de comparaison
eval_results = []
valid_algos = [
    ('K-Means', kmeans_labels),
    ('Agglomerative', agglo_labels),
    ('GMM', gmm_labels),
]
if n_dbscan_clusters > 1:
    mask = dbscan_labels != -1
    valid_algos.append(('DBSCAN', dbscan_labels))

for name, labels in valid_algos:
    try:
        sil = silhouette_score(X_pca_full, labels)
        db = davies_bouldin_score(X_pca_full, labels)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        eval_results.append({'Algorithme': name, 'N clusters': n_clust,
                              'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(db, 4)})
    except Exception as e:
        print(f"{name} : {e}")

eval_df = pd.DataFrame(eval_results).set_index('Algorithme')
print(eval_df.to_string())

In [ ]:
show_section("6. Interprétation métier des profils")

In [ ]:
# Ajouter les labels K-Means au dataframe original
df_result = df_clean.copy()
df_result['Cluster'] = kmeans_labels

# Profil moyen par cluster
profile_cols = ['Income', 'Age', 'TotalSpend', 'TotalPurchases',
                'MntWines', 'MntMeatProducts', 'NumWebPurchases',
                'NumStorePurchases', 'Recency', 'Children']
profile_cols = [c for c in profile_cols if c in df_result.columns]

cluster_profiles = df_result.groupby('Cluster')[profile_cols].mean().round(1)
print("=== Profils moyens par cluster ===")
print(cluster_profiles.T.to_string())

In [ ]:
from src.charts.segmentation import build_cluster_profiles
from src.utils import show_figure

fig = build_cluster_profiles(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_cluster_profiles.png', width=1100, height=500)


In [ ]:
# Nommer les clusters (à adapter selon les résultats)
# Exemple de nommage basé sur les profils observés :
cluster_names = {
    0: 'Clients Premium',       # Hauts revenus, fortes dépenses
    1: 'Clients Digitaux',      # Achats web élevés, jeunes
    2: 'Promo-sensibles',       # Nombreux achats promotionnels
    3: 'Clients Dormants',      # Faibles dépenses, peu actifs
}
# Adapter cluster_names selon BEST_K et les profils réels
if BEST_K <= len(cluster_names):
    df_result['Profil'] = df_result['Cluster'].map(
        {k: v for k, v in cluster_names.items() if k < BEST_K}
    )

print("Taille des clusters :")
print(df_result['Cluster'].value_counts().sort_index())

In [ ]:
from src.charts.segmentation import build_radar_profiles
from src.utils import show_figure

fig = build_radar_profiles(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_radar_profiles.png', width=1000, height=500)


In [ ]:
show_section("7. Recommandations business")

In [ ]:
from src.charts.segmentation import build_campaign_response
from src.utils import show_figure

fig = build_campaign_response(DATA_PATH, '../models')
show_figure(fig, '../reports/figures/ex2_campaign_response.png', width=900, height=450)


In [ ]:
show_section("Synthèse Exercice 2")
profiles_df = pd.DataFrame([
    {"Cluster": 0, "Profil": "Premium", "Actions": "Fidélité, offres exclusives"},
    {"Cluster": 1, "Profil": "Digitaux", "Actions": "Email, app mobile"},
    {"Cluster": 2, "Profil": "Promo-sensibles", "Actions": "Coupons ciblés"},
])
show_table_html(profiles_df.set_index("Cluster"), title="Profils (à adapter)")
show_findings_list([
    f"k optimal = {BEST_K} (Silhouette)",
    "Personnaliser les campagnes par profil cluster",
    "Recalcul mensuel avec suivi Silhouette",
], title="Recommandations stratégiques")